In [1]:
import sys
print(sys.executable)


d:\Codingapp\python.exe


In [2]:
import os
print(os.path.abspath("../bluestockmf.db"))
print(os.path.exists("../bluestockmf.db"))

d:\code\bluestockmf.db
True


In [3]:
from sqlalchemy import create_engine, inspect

engine = create_engine("sqlite:///../bluestockmf.db")
inspector = inspect(engine)
print(inspector.get_table_names())

[]


In [4]:
import os
print("Current working directory:", os.getcwd())
print("\nContents here:")
print(os.listdir())

Current working directory: d:\code\mf-analysis

Contents here:
['.git', '.gitignore', '.vscode', '05_advanced_analytics.ipynb', 'bluestockmf.db', 'bluestockmf.sqbpro', 'clean_data.py', 'Csv.ipynb', 'dashboard', 'data', 'data_dictionary.md', 'data_ingestion.py', 'day1_qualitative_summary.md', 'live_nav_fetch.py', 'load_to_db.py', 'notebooks', 'queries.sql', 'recommender.py', 'reports', 'requirements.txt', 'schema.sql', 'settings.json', 'sql']


In [5]:
from sqlalchemy import create_engine, inspect

engine = create_engine("sqlite:///bluestockmf.db")
inspector = inspect(engine)
print(inspector.get_table_names())

['dim_date', 'dim_fund', 'fact_aum', 'fact_nav', 'fact_performance', 'fact_portfolio', 'fact_sip_industry', 'fact_transactions']


In [8]:
import pandas as pd

pd.read_sql("SELECT * FROM fact_transactions", engine).to_csv("data/processed/fact_transactions_export.csv", index=False)
pd.read_sql("SELECT * FROM fact_performance", engine).to_csv("data/processed/fact_performance_export.csv", index=False)
pd.read_sql("SELECT * FROM fact_portfolio", engine).to_csv("data/processed/fact_portfolio_export.csv", index=False)
pd.read_sql("SELECT * FROM fact_sip_industry", engine).to_csv("data/processed/fact_sip_industry_export.csv", index=False)
pd.read_sql("SELECT * FROM fact_aum", engine).to_csv("data/processed/fact_aum_export.csv", index=False)
print("All 5 tables exported successfully")

All 5 tables exported successfully


In [ ]:
import pandas as pd
from sqlalchemy import create_engine

DB_PATH = r"D:\code\mf-analysis\bluestockmf.db"
engine = create_engine(f"sqlite:///{DB_PATH}")

check = pd.read_sql("SELECT * FROM fact_aum", engine)
print(check.shape)
print(check.head())

(0, 4)
Empty DataFrame
Columns: [aum_id, fund_house, date_id, aum_crore]
Index: []


In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text

DB_PATH = r"D:\code\mf-analysis\bluestockmf.db"
engine = create_engine(f"sqlite:///{DB_PATH}")

# Step 1: Load raw AUM CSV
aum_raw = pd.read_csv("data/raw/03_aum_by_fund_house.csv")
aum_raw["date"] = pd.to_datetime(aum_raw["date"])

# Step 2: Get dim_date for matching
dim_date = pd.read_sql("SELECT date_id, full_date FROM dim_date", engine)
dim_date["full_date"] = pd.to_datetime(dim_date["full_date"])

# Step 3: Match dates (nearest, since AUM is quarterly)
aum_merged = pd.merge_asof(
    aum_raw.sort_values("date"),
    dim_date.sort_values("full_date"),
    left_on="date", right_on="full_date",
    direction="nearest"
)

# Step 4: Select correct columns
aum_final = aum_merged[["fund_house", "date_id", "aum_crore"]]
print(aum_final.shape)

# Step 5: Clear and reload
with engine.connect() as conn:
    conn.execute(text("DELETE FROM fact_aum"))
    conn.commit()

aum_final.to_sql("fact_aum", engine, if_exists="append", index=False)

# Step 6: Verify
print(pd.read_sql("SELECT COUNT(*) FROM fact_aum", engine))

(90, 3)
   COUNT(*)
0        90
